# Soft-Matching Distance on the MNIST models

Appendix table: does the **Soft-Matching Distance** (SMD; Khosla & Williams, AISTATS 2024) tell the
clustered MNIST models ($\lambda{=}10$) apart from the baseline models ($\lambda{=}0$)? SMD matches
single-neuron tuning vectors across two networks with optimal transport, so it depends on
neuron-level tuning and not only on population geometry.

**Comparisons** (20 seeds per condition; the checkpoints are the ones behind Table 2)

* **(a) cross-condition:** baseline seed $i$ vs clustered seed $i$, $i = 0,\dots,19$ (20 pairs)
* **(a′) cross-condition, different seed:** baseline seed $i$ vs clustered seed $(i+1) \bmod 20$ (20 pairs)
* **(b) seed reference:** baseline seed $i$ vs baseline seed $(i+1) \bmod 20$ (20 pairs), for ordinary seed-to-seed variation

**Why (a′) is needed.** `train_or_load_cnn` calls `torch.manual_seed(seed)` before building the CNN,
so baseline seed $i$ and clustered seed $i$ start from **identical initial weights**. Pair (a)
shares its initialisation and pair (b) does not. A neuron-level metric like SMD can pick up that
shared starting point, while RSA/CKA/Procrustes cannot see unit identity at all. In (a′) vs (b) the
second model (seed $i{+}1$) has the same initialisation in both pairs, so the only difference is its
training loss. **The confound-free test is (a′) vs (b).** GW is cached only for (a) and (b); it
compares within-population distances and does not use unit identity.

**Responses.** The 800 flatten-layer units respond to one fixed set of 500 test images: the first
50 test images of each digit, in test-set order. This is the same set `build_tensor4d` uses for
Table 2, and a check below asserts that they match.

**SMD.** Let unit $i$ of network $A$ have tuning vector $\mathbf{x}_i \in \mathbb{R}^{M}$ ($M = 500$
images). Each unit is z-scored across images. The cost is
$C_{ij} = \tfrac{1}{M}\lVert \mathbf{z}^A_i - \mathbf{z}^B_j \rVert^2$, and exact OT (`ot.emd`) is
solved with uniform marginals:

$$\mathrm{SMD}(A,B) = \Big(\min_{P \in \Pi(\mathbf{1}/N_A,\,\mathbf{1}/N_B)} \sum_{ij} P_{ij} C_{ij}\Big)^{1/2}.$$

The $1/M$ factor is a constant, so the value is the prompt's $\sqrt{\sum P\odot C}$ divided by
$\sqrt{M}$, which leaves every test unaffected. For z-scored vectors
$C_{ij} = 2(1-r_{ij})$, where $r_{ij}$ is the Pearson correlation of the two tuning vectors. So
$\mathrm{SMD} \in [0, 2]$, and $\bar r = 1 - \mathrm{SMD}^2/2$ is the transport-weighted mean
tuning correlation of matched units. $\mathrm{SMD}=\sqrt2$ means matched units are uncorrelated.

**Dead units** (zero variance over the 500 images) have no defined tuning shape. The primary
variant drops them, so the marginals are uniform over each network's live units. Two variants are
reported as checks: *eps* keeps all 800 units and z-scores with $\sigma+\varepsilon$, which sends dead
units to the zero vector; *raw* skips z-scoring, so it depends on amplitude.

**Other metrics** for the same pairs. RSA, linear CKA and Procrustes $R^2$ are recomputed here with
the paper's functions (`src.subpop_utils`). GW similarity is **read** from the Table 2 cache
(`fig/paper/cache/table1_mnist_results_v2.pkl`, written by notebook 13) and not recomputed.

**Test:** two-sided Wilcoxon signed-rank, (a′) vs (b) and (a) vs (b), paired by baseline seed $i$, with Holm
correction across the rows of the table. Caveat: the (b) pairs share models (each baseline seed
appears in two of them), so the 20 (b) values are not fully independent.

In [1]:
import os, sys, time, pickle, itertools
os.environ.setdefault('OMP_NUM_THREADS', '1')        # keep the laptop responsive
import numpy as np
import pandas as pd
import torch
import ot
from scipy.stats import wilcoxon
from scipy.spatial import procrustes
from scipy.optimize import linear_sum_assignment

sys.path.insert(0, '..')
from src.mnist_utils import load_cnn_checkpoint, load_mnist_data, build_tensor4d
from src.subpop_utils import rdm_correlation, linear_cka, compute_decoding_manifold

torch.set_num_threads(2)
BASEDIR     = '..'
CKPT_DIR    = os.path.join(BASEDIR, 'data', 'mnist_manifold')
T1_CACHE    = os.path.join(BASEDIR, 'fig', 'paper', 'cache', 'table1_mnist_results_v2.pkl')
OUT_CACHE   = os.path.join(BASEDIR, 'fig', 'paper', 'cache', 'smd_mnist.pkl')
OUT_TABLE   = os.path.join(BASEDIR, 'fig', 'paper', 'tables', 'tab_smd_mnist.tex')
SEEDS       = list(range(20))
LAMBDA      = 10
N_PER_CLASS = 50
DEAD_TOL    = 1e-8          # a unit is "dead" if its std over the 500 images is below this
EPS         = 1e-8
print('POT', ot.__version__, '| torch', torch.__version__)

POT 0.9.6.post1 | torch 2.9.1


## 1. Fixed image set and flatten-layer responses

In [2]:
_, test_loader = load_mnist_data(batch_size=256)          # shuffle=False → fixed order
test_ds = test_loader.dataset
labels  = test_ds.targets.numpy()

# first 50 test images of each digit, in test-set order, grouped by class (= build_tensor4d)
IMG_IDX = np.concatenate([np.flatnonzero(labels == c)[:N_PER_CLASS] for c in range(10)])
X_IMG   = torch.stack([test_ds[i][0] for i in IMG_IDX])  # (500, 1, 28, 28), normalised
Y_IMG   = labels[IMG_IDX]
assert len(IMG_IDX) == 500 and np.array_equal(Y_IMG, np.repeat(np.arange(10), N_PER_CLASS))

def ckpt(lam, seed):
    name = f'cnn_baseline_lam0.0_e5_s{seed}.pt' if lam == 0 else \
           f'cnn_cluster_pref_stim_lam{lam}_e5_s{seed}.pt'
    return os.path.join(CKPT_DIR, name)

def responses(model):
    """(800 units, 500 images) flatten-layer activations."""
    with torch.no_grad():
        return model._h2(X_IMG).numpy().T.astype(np.float64)

def full_test_acc(model):
    correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            correct += (model(xb).argmax(1) == yb).sum().item()
    return correct / len(test_ds)

t0 = time.time()
R, ACC = {}, {}
for lam in (0, LAMBDA):
    for s in SEEDS:
        m = load_cnn_checkpoint(ckpt(lam, s))
        R[(lam, s)]   = responses(m)
        ACC[(lam, s)] = full_test_acc(m)
    print(f'lambda={lam:>2}: 20 models loaded, test acc {np.mean([ACC[(lam, s)] for s in SEEDS]):.4f}'
          f'  ({time.time() - t0:.0f}s)')

lambda= 0: 20 models loaded, test acc 0.9853  (17s)


lambda=10: 20 models loaded, test acc 0.9828  (34s)


In [3]:
# Sanity check: these are exactly the responses the paper's RSA/CKA/Procrustes (Table 2) use.
for lam in (0, LAMBDA):
    for s in (0, 19):
        t4d, _ = build_tensor4d(load_cnn_checkpoint(ckpt(lam, s)), test_loader, n_per_class=N_PER_CLASS)
        assert np.allclose(t4d[:, :, :, 0].reshape(800, -1), R[(lam, s)], atol=1e-5), (lam, s)
print('responses identical to build_tensor4d (Table 2 input)  ✓')

def to_t4d(resp):
    """(800, 500) → (800, 10, 50, 1), the tensor layout of src.subpop_utils."""
    return resp.reshape(resp.shape[0], 10, N_PER_CLASS, 1)

dead = {k: int((v.std(axis=1) < DEAD_TOL).sum()) for k, v in R.items()}
for lam in (0, LAMBDA):
    d = np.array([dead[(lam, s)] for s in SEEDS])
    print(f'dead units (lambda={lam:>2}): mean {d.mean():.1f}, range {d.min()}–{d.max()} of 800')

responses identical to build_tensor4d (Table 2 input)  ✓
dead units (lambda= 0): mean 6.6, range 0–25 of 800
dead units (lambda=10): mean 3.0, range 0–25 of 800


## 2. Soft-Matching Distance and checks with known answers

In [4]:
def smd(RA, RB, mode='z'):
    """Soft-Matching Distance between two (units × images) response matrices.

    mode='z'   : z-score each unit across images, drop dead units (primary)
    mode='eps' : z-score with std + EPS, keep all units (dead units -> zero vector)
    mode='raw' : raw activations, no z-scoring (amplitude-sensitive)
    Cost = squared Euclidean distance / n_images; exact EMD with uniform marginals.
    Returns (distance, coupling P, cost C).
    """
    A, B = RA.astype(np.float64), RB.astype(np.float64)
    if mode in ('z', 'eps'):
        if mode == 'z':
            A = A[A.std(axis=1) >= DEAD_TOL]
            B = B[B.std(axis=1) >= DEAD_TOL]
        A = (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + (EPS if mode == 'eps' else 0.0))
        B = (B - B.mean(1, keepdims=True)) / (B.std(1, keepdims=True) + (EPS if mode == 'eps' else 0.0))
    elif mode != 'raw':
        raise ValueError(mode)
    C = ot.dist(A, B, metric='sqeuclidean') / A.shape[1]
    a = np.full(A.shape[0], 1.0 / A.shape[0])
    b = np.full(B.shape[0], 1.0 / B.shape[0])
    P = ot.emd(a, b, C, numItermax=10_000_000)
    return float(np.sqrt(max(0.0, np.sum(P * C)))), P, C

# ── Known-answer checks ──────────────────────────────────────────────────────
RA, RB = R[(0, 0)], R[(LAMBDA, 0)]
rng = np.random.default_rng(0)

# (i) SMD is invariant to relabelling the units: SMD(A, A permuted) = 0
for mode in ('z', 'eps', 'raw'):
    d0, _, _ = smd(RA, RA[rng.permutation(800)], mode)
    assert d0 < 1e-6, (mode, d0)
print('(i)   SMD(A, permuted A) = 0 for all variants  ✓')

# (ii) Equal sizes with uniform weights: the OT optimum is a permutation (Birkhoff), so EMD must
#      equal the Hungarian assignment optimum.
for mode in ('eps', 'raw'):
    d, P, C = smd(RA, RB, mode)
    r, c = linear_sum_assignment(C)
    assert np.isclose(d, np.sqrt(C[r, c].mean()), rtol=1e-9), mode
    assert np.isclose(P.sum(), 1) and np.allclose(P.sum(1), 1 / 800) and np.allclose(P.sum(0), 1 / 800)
print('(ii)  EMD value == Hungarian assignment optimum; marginals exact  ✓')

# (iii) symmetry, and cost identity C = 2(1 - Pearson r) for z-scored units
d_ab, _, C = smd(RA, RB, 'z')
d_ba, _, _ = smd(RB, RA, 'z')
assert np.isclose(d_ab, d_ba, rtol=1e-9)
la, lb = RA[RA.std(1) >= DEAD_TOL], RB[RB.std(1) >= DEAD_TOL]
r_ab = np.corrcoef(la, lb)[:len(la), len(la):]
assert np.allclose(C, 2 * (1 - r_ab), atol=1e-8)
print('(iii) SMD symmetric; z-scored cost = 2(1 - r)  ✓')

# (iv) Reference scale: scramble each unit's responses across images → matched tuning uncorrelated
RB_scr = np.array([row[rng.permutation(500)] for row in RB])
print(f'(iv)  seed-0 SMD(base, clustered) = {d_ab:.3f};  vs image-scrambled clustered = '
      f'{smd(RA, RB_scr, "z")[0]:.3f}  (sqrt 2 = {np.sqrt(2):.3f} = uncorrelated)')

(i)   SMD(A, permuted A) = 0 for all variants  ✓


(ii)  EMD value == Hungarian assignment optimum; marginals exact  ✓


(iii) SMD symmetric; z-scored cost = 2(1 - r)  ✓
(iv)  seed-0 SMD(base, clustered) = 0.671;  vs image-scrambled clustered = 1.318  (sqrt 2 = 1.414 = uncorrelated)


## 3. All pairs: SMD, RSA, CKA, Procrustes $R^2$

In [5]:
def procrustes_r2(RA, RB):
    """As in Table 2: 1 - disparity between the 3-PC decoding manifolds."""
    ca, _ = compute_decoding_manifold(to_t4d(RA), n_components=3)
    cb, _ = compute_decoding_manifold(to_t4d(RB), n_components=3)
    return 1.0 - procrustes(ca, cb)[2]

PAIRS = {'cross': [((0, s), (LAMBDA, s)) for s in SEEDS],                       # (a)
         'cross_ds': [((0, s), (LAMBDA, (s + 1) % len(SEEDS))) for s in SEEDS],  # (a')
         'base':  [((0, s), (0, (s + 1) % len(SEEDS))) for s in SEEDS]}        # (b)

res = {k: {m: [] for m in ('smd_z', 'smd_eps', 'smd_raw', 'rbar', 'rsa', 'cka', 'proc_r2', 'id_corr')}
       for k in PAIRS}
t0 = time.time()
for kind, pairs in PAIRS.items():
    for ka, kb in pairs:
        RA, RB = R[ka], R[kb]
        d_z, _, _ = smd(RA, RB, 'z')
        res[kind]['smd_z'].append(d_z)
        res[kind]['rbar'].append(1 - d_z ** 2 / 2)
        res[kind]['smd_eps'].append(smd(RA, RB, 'eps')[0])
        res[kind]['smd_raw'].append(smd(RA, RB, 'raw')[0])
        res[kind]['rsa'].append(rdm_correlation(to_t4d(RA), to_t4d(RB)))
        res[kind]['cka'].append(linear_cka(to_t4d(RA), to_t4d(RB)))
        res[kind]['proc_r2'].append(procrustes_r2(RA, RB))
        # shared-initialisation diagnostic: tuning correlation of unit k in A with the SAME unit k in B
        live = (RA.std(1) >= DEAD_TOL) & (RB.std(1) >= DEAD_TOL)
        za = (RA[live] - RA[live].mean(1, keepdims=True)) / RA[live].std(1, keepdims=True)
        zb = (RB[live] - RB[live].mean(1, keepdims=True)) / RB[live].std(1, keepdims=True)
        res[kind]['id_corr'].append(float(np.mean(np.sum(za * zb, axis=1) / za.shape[1])))
    print(f'{kind:>8}: 20 pairs done ({time.time() - t0:.0f}s)')
res = {k: {m: np.asarray(v) for m, v in d.items()} for k, d in res.items()}

   cross: 20 pairs done (5s)


cross_ds: 20 pairs done (11s)


    base: 20 pairs done (16s)


## 4. GW similarity from the Table 2 cache

`gw_ian[i]` is baseline seed $i$ vs clustered seed $i$. `within_base_gw_ian` covers all 190
baseline pairs $(s_1<s_2)$ in `itertools.combinations` order (notebook 13, cell "Table 1"). To check
that the cache came from these checkpoints, its cross-condition RSA entries are compared with the
values recomputed above.

In [6]:
with open(T1_CACHE, 'rb') as f:
    t1 = pickle.load(f)
assert len(t1['gw_ian']) == 20 and len(t1['within_base_gw_ian']) == 190

pair_index = {p: i for i, p in enumerate(itertools.combinations(SEEDS, 2))}
res['cross']['gw'] = np.asarray(t1['gw_ian'], dtype=float)
res['base']['gw']  = np.asarray([t1['within_base_gw_ian'][pair_index[tuple(sorted((s, (s + 1) % 20)))]]
                                 for s in SEEDS], dtype=float)

# Provenance check: each recomputed cross-condition RSA must appear in the cache.
cached_rsa = np.asarray(t1[LAMBDA]['rsa'], dtype=float)
match = [bool(np.any(np.isclose(cached_rsa, v, atol=1e-6))) for v in res['cross']['rsa']]
print(f'recomputed cross RSA found in cache for {sum(match)}/20 seeds')
print(f'cache holds {len(cached_rsa)} cross-RSA entries; entries 0-9 == 10-19: '
      f'{np.allclose(cached_rsa[:10], cached_rsa[10:20])}')

recomputed cross RSA found in cache for 20/20 seeds
cache holds 30 cross-RSA entries; entries 0-9 == 10-19: True


## 5. Table

In [7]:
def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (len(p) - rank) * p[i])); adj[i] = run
    return adj

# (key, label, higher value means more similar?)
ROWS = [('rsa',     'RSA',                         True),
        ('cka',     'Linear CKA',                  True),
        ('proc_r2', 'Procrustes $R^2$',            True),
        ('smd_z',   'SMD (z-scored)',              False),
        ('gw',      'GW similarity',               True)]
CHECKS = [('smd_eps', 'SMD (z-scored, eps, all 800 units)', False),
          ('smd_raw', 'SMD (raw, no z-scoring)',            False),
          ('rbar',    r'matched tuning corr. $\bar r$',      True)]

def test(key, sim, kind):
    """(a)/(a') vs (b): Wilcoxon p, #seeds where the cross pair is less similar, effect in SD(b)."""
    a, b = res[kind][key], res['base'][key]
    p = wilcoxon(a, b, alternative='two-sided').pvalue
    n_less = int(np.sum(a < b) if sim else np.sum(a > b))
    return p, n_less, (a.mean() - b.mean()) / b.std(ddof=1)

def msem(x):
    return x.mean(), x.std(ddof=1) / np.sqrt(len(x))

def summarise(rows):
    out = []
    for key, label, sim in rows:
        row = {'metric': label}
        for kind, tag in (('base', 'b'), ('cross', 'a'), ('cross_ds', 'a2')):
            if key in res[kind]:
                row[f'{tag}_mean'], row[f'{tag}_sem'] = msem(res[kind][key])
        for kind, tag in (('cross', 'a'), ('cross_ds', 'a2')):
            if key in res[kind]:
                row[f'p_{tag}'], row[f'nless_{tag}'], row[f'eff_{tag}'] = test(key, sim, kind)
        out.append(row)
    return pd.DataFrame(out)

main = summarise(ROWS)
chk  = summarise(CHECKS + [('id_corr', 'same-index unit tuning corr.', True)])
# Holm across the main rows: the (a') tests plus GW's (a) test, the only GW test available
fam = [(i, 'p_a2') if pd.notna(r.get('p_a2')) else (i, 'p_a') for i, r in main.iterrows()]
adj = holm([main.loc[i, c] for i, c in fam])
for (i, c), q in zip(fam, adj):
    main.loc[i, 'p_holm'] = q
table = pd.concat([main, chk], ignore_index=True)
pd.set_option('display.float_format', lambda v: f'{v:.4g}')
pd.set_option('display.width', 250)
display(table)

,metric,b_mean,b_sem,a_mean,a_sem,a2_mean,a2_sem,p_a,nless_a,eff_a,p_a2,nless_a2,eff_a2,p_holm
0,RSA,0.9551,0.003909,0.9618,0.002833,0.925,0.00595,0.1327,6,0.3814,1.907e-06,20,-1.719,9.537e-06
1,Linear CKA,0.9747,0.001693,0.9763,0.001641,0.9561,0.002543,0.6215,10,0.2,1.907e-06,20,-2.463,9.537e-06
2,Procrustes $R^2$,0.9476,0.00752,0.943,0.01048,0.9166,0.009979,0.7285,9,-0.1369,0.00486,17,-0.9219,0.00486
3,SMD (z-scored),0.7107,0.005389,0.6823,0.005051,0.772,0.005336,9.537e-06,1,-1.181,1.907e-06,20,2.543,9.537e-06
4,GW similarity,0.957,0.002399,0.5016,0.02233,NaN,NaN,1.907e-06,20,-42.45,NaN,NaN,NaN,9.537e-06
5,"SMD (z-scored, eps, all 800 units)",0.7097,0.005414,0.6818,0.005061,0.7708,0.005278,2.67e-05,1,-1.152,1.907e-06,20,2.524,NaN
6,"SMD (raw, no z-scoring)",1.1,0.01197,1.034,0.01139,1.198,0.0137,0.0001049,3,-1.245,1.907e-06,20,1.824,NaN
7,matched tuning corr. $\bar r$,0.7471,0.003798,0.767,0.003465,0.7017,0.004078,9.537e-06,1,1.17,1.907e-06,20,-2.674,NaN
8,same-index unit tuning corr.,0.1641,0.01075,0.6993,0.006602,0.1629,0.0107,1.907e-06,0,11.13,0.8695,10,-0.0247,NaN


In [8]:
def pm(m, s):
    return f'${m:.3f} \\pm {s:.3f}$'

def pfmt(p):
    if p < 1e-3:
        mant, ex = f'{p:.1e}'.split('e')
        return f'${mant}\\times 10^{{{int(ex)}}}$'
    return f'${p:.3f}$'

def cell(r, tag):
    m = r.get(f'{tag}_mean')
    return '--' if m is None or pd.isna(m) else pm(m, r[f'{tag}_sem'])

lines = [r'\begin{tabular}{l c c c c c}', r'\toprule',
         r'Metric & Base$_i$ vs.\ Base$_{i+1}$ & Base$_i$ vs.\ Clust$_i$ & Base$_i$ vs.\ Clust$_{i+1}$'
         r' & $p$ (Holm) & seeds \\', r'\midrule']
for _, r in main.iterrows():
    use_a2 = pd.notna(r.get('p_a2'))
    nless = int(r['nless_a2'] if use_a2 else r['nless_a'])
    lines.append(f"{r.metric} & {cell(r, 'b')} & {cell(r, 'a')} & {cell(r, 'a2')} & "
                 f"{pfmt(r.p_holm)}{'' if use_a2 else '$^\\ddagger$'} & {nless}/20 \\\\")
lines.append(r'\midrule')
for _, r in chk.iterrows():
    lines.append(f"\\quad {r.metric} & {cell(r, 'b')} & {cell(r, 'a')} & {cell(r, 'a2')} & "
                 f"{pfmt(r.p_a2)}$^\\dagger$ & {int(r.nless_a2)}/20 \\\\")
lines += [r'\bottomrule', r'\end{tabular}']
tex = '\n'.join(lines)
os.makedirs(os.path.dirname(OUT_TABLE), exist_ok=True)
with open(OUT_TABLE, 'w') as f:
    f.write(tex + '\n')
with open(OUT_CACHE, 'wb') as f:
    pickle.dump({'res': res, 'table': table, 'img_idx': IMG_IDX, 'acc': ACC, 'dead': dead}, f)
print(tex)
print(f"\np, seeds: Base_i vs Clust_(i+1) against Base_i vs Base_(i+1); ‡ GW: only Base_i vs Clust_i is cached.")
print(f'† raw (uncorrected) p for the check rows.  Written: {OUT_TABLE}, {OUT_CACHE}')
for _, r in main.iterrows():
    if pd.notna(r.get('p_a2')):
        print(f"{r.metric:>18}: same-seed pair (a) vs (b): p = {r.p_a:.2e}, cross less similar in {int(r.nless_a)}/20")

\begin{tabular}{l c c c c c}
\toprule
Metric & Base$_i$ vs.\ Base$_{i+1}$ & Base$_i$ vs.\ Clust$_i$ & Base$_i$ vs.\ Clust$_{i+1}$ & $p$ (Holm) & seeds \\
\midrule
RSA & $0.955 \pm 0.004$ & $0.962 \pm 0.003$ & $0.925 \pm 0.006$ & $9.5\times 10^{-6}$ & 20/20 \\
Linear CKA & $0.975 \pm 0.002$ & $0.976 \pm 0.002$ & $0.956 \pm 0.003$ & $9.5\times 10^{-6}$ & 20/20 \\
Procrustes $R^2$ & $0.948 \pm 0.008$ & $0.943 \pm 0.010$ & $0.917 \pm 0.010$ & $0.005$ & 17/20 \\
SMD (z-scored) & $0.711 \pm 0.005$ & $0.682 \pm 0.005$ & $0.772 \pm 0.005$ & $9.5\times 10^{-6}$ & 20/20 \\
GW similarity & $0.957 \pm 0.002$ & $0.502 \pm 0.022$ & -- & $9.5\times 10^{-6}$$^\ddagger$ & 20/20 \\
\midrule
\quad SMD (z-scored, eps, all 800 units) & $0.710 \pm 0.005$ & $0.682 \pm 0.005$ & $0.771 \pm 0.005$ & $1.9\times 10^{-6}$$^\dagger$ & 20/20 \\
\quad SMD (raw, no z-scoring) & $1.100 \pm 0.012$ & $1.034 \pm 0.011$ & $1.198 \pm 0.014$ & $1.9\times 10^{-6}$$^\dagger$ & 20/20 \\
\quad matched tuning corr. $\bar r$ & $0.

## 6. Amplitude check for the raw variant

Raw SMD is sensitive to response magnitude. If the clustered models' units fire at a different
overall scale, raw SMD can separate the conditions without any change in tuning shape. The cell
below reports the per-unit response scale for each condition.

In [9]:
for lam in (0, LAMBDA):
    mu = np.array([R[(lam, s)].mean() for s in SEEDS])
    sd = np.array([np.median(R[(lam, s)].std(axis=1)) for s in SEEDS])
    print(f'lambda={lam:>2}: mean activation {mu.mean():.3f} ± {mu.std(ddof=1)/np.sqrt(20):.3f}, '
          f'median unit std {sd.mean():.3f} ± {sd.std(ddof=1)/np.sqrt(20):.3f}')

lambda= 0: mean activation 1.212 ± 0.014, median unit std 1.227 ± 0.013
lambda=10: mean activation 1.023 ± 0.016, median unit std 1.322 ± 0.017
